# Adversarial dosage sweep — does retraining ever beat this attacker?

**The question.** Our tabular headline is ASR = 1.000 at every round: three rounds of
adversarial retraining prevented zero evasions. The writeup explains that as a dosage
problem — 400 adversarial rows folded unweighted into 196,001, about 0.2% of the
training mass. That is a *hypothesis*, and shipping a hypothesis where a result belongs
is the thing this project spends its methodology section arguing against.

This tests it. Each arm re-runs the loop with adversarial rows carrying
`sample_weight = w`, sweeping `w` wide enough to bracket the transition: at `w=1` they
are 0.2% of training mass, at `w=5000` they outweigh the entire legitimate trainset.

**All three outcomes are publishable**, which is why it earns the compute:

| Outcome | What we report |
|---|---|
| ASR falls at some dosage | The dosage where the defence starts working, and its PR-AUC cost |
| ASR never falls | Our own "dosage was too small" excuse is refuted by experiment |
| ASR falls only as PR-AUC collapses | The defence is real but not worth buying |

---

## Setup — two inputs, no internet needed

1. **Add the data.** *Add Input* → Datasets → search `kartik2112/fraud-detection` → Add.
2. **Add the code.** Build it with `python scripts/make_kaggle_bundle.py`, then
   *Add Input* → *Upload* → drop in `kaggle_code.zip` (Kaggle unzips it for you).
   Name it anything. Cell 1 prints the git sha it was built from, so a stale
   bundle is visible immediately rather than after four hours of compute.
3. **Save Version → Save & Run All (Commit)** — that is what runs it overnight,
   detached from your browser. An interactive session dies when the tab closes.

Internet can stay **off**. Nothing here pip-installs: the sweep pulls only pandas,
numpy, sklearn, scipy and xgboost, all of which Kaggle preinstalls.

Expect roughly 2–4 hours on the full 1.85M rows, inside Kaggle's 12h commit limit.
Results are written after **every arm**, so a run that dies in its last arm still
leaves the arms that finished.


In [ ]:
# --- 1. code ---------------------------------------------------------------------
# Prefer the uploaded code dataset; fall back to cloning only if Internet happens to
# be on. Either way the code is COPIED into /kaggle/working, because the package
# derives its project root from its own __file__ and then writes the interim parquet
# under it -- which fails on the read-only /kaggle/input mount.
import glob, os, shutil, subprocess, sys

DEST = '/kaggle/working/repo/src'
os.makedirs(DEST, exist_ok=True)

src = None
for cand in glob.glob('/kaggle/input/*/**/adversarial_payments', recursive=True):
    if os.path.isdir(cand):
        src = os.path.dirname(cand)
        break

if src:
    print('found uploaded code at', src)
    shutil.copytree(os.path.join(src, 'adversarial_payments'),
                    os.path.join(DEST, 'adversarial_payments'), dirs_exist_ok=True)
    sweep = glob.glob(os.path.join(src, '**', 'run_dosage_sweep.py'), recursive=True)
    assert sweep, 'run_dosage_sweep.py missing from the uploaded zip'
    shutil.copy(sweep[0], os.path.join(DEST, 'run_dosage_sweep.py'))
else:
    print('no uploaded code found; trying git clone (needs Internet ON)')
    r = subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/Aditya-Patil27/mastercard-adversarial-payments.git',
                        '/kaggle/working/clone'], capture_output=True, text=True)
    assert r.returncode == 0, (
        'No code dataset AND clone failed. Upload kaggle_code.zip as an input, '
        'or enable Internet in Settings.\n' + r.stderr[-400:])
    shutil.copytree('/kaggle/working/clone/src/adversarial_payments',
                    os.path.join(DEST, 'adversarial_payments'), dirs_exist_ok=True)
    shutil.copy('/kaggle/working/clone/scripts/run_dosage_sweep.py',
                os.path.join(DEST, 'run_dosage_sweep.py'))

sys.path.insert(0, DEST)
import adversarial_payments.config as C
print('package root (must be writable):', C.ROOT)
assert str(C.ROOT).startswith('/kaggle/working'), C.ROOT

# The bundle stamps its own git sha. A zip uploaded once and forgotten is the
# obvious way to lose a night to code that no longer matches the repo, so it
# announces itself here rather than in numbers nobody can reproduce.
import json
info = glob.glob(os.path.join(src or '', '**', 'BUNDLE_INFO.json'), recursive=True)
if info:
    meta = json.load(open(info[0]))
    print(f"bundle built from {meta['git_sha']} at {meta['built_at']}" + ('  [DIRTY]' if meta.get('dirty') else ''))
else:
    print('bundle has no BUNDLE_INFO.json -- rebuild with scripts/make_kaggle_bundle.py')


In [ ]:
# --- 2. data ---------------------------------------------------------------------
# Find the dataset by its SCHEMA, not by its path.
#
# Kaggle mounts a DATASET at /kaggle/input/<slug>/ but a NOTEBOOK's output at
# /kaggle/input/notebooks/<user>/<slug>/. Matching a directory name containing "fraud"
# would happily accept either, or a lookalike dataset with different columns, and the
# failure would not surface until hours later. Checking for the 22 raw Sparkov columns is
# the same discipline the rest of the repo uses: verify the thing, never trust its label.
#
# kagglehub.dataset_download() does NOT work here -- it needs internet, which is off.
# "Add Input -> Datasets" is the Kaggle-native equivalent and needs no network.
import glob
import os

import pandas as pd

from adversarial_payments.data.synthetic import RAW_COLUMNS

REQUIRED = set(RAW_COLUMNS)
csvs = sorted(glob.glob("/kaggle/input/**/*.csv", recursive=True))
print(f"{len(csvs)} csv(s) under /kaggle/input")

hit = None
for path in csvs:
    try:
        cols = set(pd.read_csv(path, nrows=0).columns)
    except Exception:
        continue
    if REQUIRED <= cols:
        hit = path
        break

if hit is None:
    print("\nNo CSV carries the Sparkov schema. What is actually mounted:")
    for path in csvs[:25]:
        try:
            head = list(pd.read_csv(path, nrows=0).columns)[:6]
        except Exception as exc:
            head = [f"<unreadable: {type(exc).__name__}>"]
        print(f"  {path}")
        print(f"      {head}")
    if not csvs:
        print("  (nothing)")
    raise SystemExit(
        "Add Input -> Datasets -> kartik2112/fraud-detection.  "
        "A NOTEBOOK of the same name is not the dataset: notebook outputs mount under "
        "/kaggle/input/notebooks/... and do not contain the raw CSVs."
    )

os.environ["SPARKOV_CSV_DIR"] = os.path.dirname(hit)
print("matched Sparkov schema in:", hit)
print("SPARKOV_CSV_DIR         :", os.environ["SPARKOV_CSV_DIR"])


In [ ]:
# --- 3. sanity check before committing hours -------------------------------------
# Two minutes that prove the data parses and the schema contract holds. If something is
# wrong it fails here, not four hours into an unattended run.
from adversarial_payments.data.load import load_features, read_provenance
from adversarial_payments.schema import TARGET

probe = load_features(sample_rows=20_000)
print(f"{len(probe):,} rows, {int(probe[TARGET].sum()):,} fraud "
      f"({probe[TARGET].mean():.4%} base rate)")
assert probe[TARGET].sum() > 0, "no positives -- wrong dataset or a parsing failure"

# The loader falls back to deterministic synthetic data when the real dataset cannot be
# read, and records that fact rather than pretending. Unattended, that fallback is the
# worst available outcome: a full overnight sweep computed on fabricated rows that looks
# exactly like a real one. Fail loudly here instead.
prov = read_provenance()
print("provenance source:", prov["source"], "| rows:", f"{prov['n_rows']:,}")
assert prov["source"] == "kaggle", (
    f"fell back to {prov['source']} data -- refusing to sweep on it. Check the input."
)


In [ ]:
# --- 4. the sweep ----------------------------------------------------------------
# Run in-process rather than via !python: a subprocess would not inherit the sys.path
# set above, and exporting PYTHONPATH to paper over that is one more thing to get
# wrong at 3am. Output streams live either way.
#
# ROWS = 0 uses all 1.85M rows. Lower to 400_000 for a faster answer; arms stay
# comparable either way, since every arm sees the identical split and round 0.
ROWS     = 0
ROUNDS   = 3
ATTEMPTS = 800
WEIGHTS  = ['1', '10', '50', '200', '1000', '5000']
OUT      = '/kaggle/working/dosage_sweep.json'

import run_dosage_sweep

run_dosage_sweep.main([
    '--rows', str(ROWS), '--rounds', str(ROUNDS), '--attempts', str(ATTEMPTS),
    '--weights', *WEIGHTS, '--out', OUT,
])


In [ ]:
# --- 5. read the result ----------------------------------------------------------
import json, pandas as pd

data = json.load(open(OUT))
df = pd.DataFrame([r for arm in data['arms'] for r in arm['rounds']])

print(f"rows={data['rows']:,}  train={data['n_train']:,}  test={data['n_test']:,}")
print(f"attempts/round={data['attempts_per_round']}  fpr_budget={data['fpr_budget']}")
print()
print(df.pivot(index='adversarial_weight', columns='round',
               values=['asr', 'pr_auc']).to_string())


In [ ]:
# --- 6. the answer, stated plainly -----------------------------------------------
final = df[df['round'] == df['round'].max()]
base  = float(df[df['round'] == 0]['asr'].iloc[0])
best  = final.loc[final['asr'].idxmin()]
clean = float(df[(df['round'] == 0)]['pr_auc'].iloc[0])

print(f'round-0 ASR (undefended)      : {base:.3f}   PR-AUC {clean:.4f}')
print(f'best final ASR across dosages : {best["asr"]:.3f} at weight {best["adversarial_weight"]:g}')
print(f'  its PR-AUC                  : {best["pr_auc"]:.4f} ({100*(best["pr_auc"]-clean)/clean:+.1f}% vs round 0)')
print()
if best['asr'] >= base - 1e-9:
    print('VERDICT: ASR never fell, at any dosage tried -- including weights where the')
    print('adversarial rows outweigh the entire legitimate trainset.')
    print('Our own "the dosage was too small" explanation is REFUTED by our own experiment.')
    print('Report: adversarial retraining does not defeat this attacker at any dosage we')
    print('can afford. Say it before a judge finds it.')
elif best['pr_auc'] < 0.9 * clean:
    print('VERDICT: ASR fell, but only where PR-AUC collapsed with it.')
    print('That is a defence that works by breaking the detector -- report both numbers')
    print('together or the result is misleading.')
else:
    print('VERDICT: ASR fell at a tolerable PR-AUC cost. This is the co-evolution result')
    print('the project set out to find. Report the dosage, the ASR AND the PR-AUC cost.')


## After it finishes

Download `dosage_sweep.json` from the notebook Output, drop it into
`artifacts/attack/`, and commit. Whatever it says goes into the writeup as-is —
including, and especially, the outcome where our own stated explanation turns out to
be wrong. That outcome is worth more than a flattering curve, because it is the one a
judge cannot find before we do.
